# Exploración: Google News (RSS) por nombre o palabra clave

1. **Búsqueda** de dos empresas, una persona y un tema.
2. **Campos** que entrega cada noticia y con qué frecuencia vienen llenos.
3. **Ruido**: cuántos títulos contienen lo buscado.

Petición HTTP simple: sin navegador ni captcha. Ante un error o bloqueo **se detiene**; no reintenta.

> **Uso:** el feed declara que es solo para lectores personales y no comerciales (ver celda 1). Esta es una
> exploración puntual; integrarlo requiere autorización u otra fuente.

Una coincidencia de nombre **no** identifica a una persona ni a una empresa: lo confirma un analista.
Limpia las salidas antes de hacer commit.

In [ ]:
import re, html, time, unicodedata
import xml.etree.ElementTree as ET
from collections import Counter
from email.utils import parsedate_to_datetime

import httpx

FEED = "https://news.google.com/rss/search"
PERU = {"hl": "es-419", "gl": "PE", "ceid": "PE:es-419"}

# etiqueta -> (consulta, términos que deben aparecer en el título para contarla como coincidencia)
CASOS = {
    "Backus": ('"Backus"', ["backus"]),
    "Alicorp": ('"Alicorp"', ["alicorp"]),
    "Persona: frase exacta": ('"Luis Fernando Chumbes Ramos"', ["luis fernando chumbes ramos"]),
    "Persona: apellidos primero": ('"Chumbes Ramos Luis Fernando"', ["chumbes ramos luis fernando"]),
    "Persona: nombre corto": ('"Luis Chumbes"', ["luis chumbes"]),
    "Persona: sin comillas": ("Luis Fernando Chumbes Ramos", ["luis fernando chumbes ramos"]),
    "Backus + Indecopi": ('"Backus" Indecopi', ["backus", "indecopi"]),
    "Alicorp + Indecopi": ('"Alicorp" Indecopi', ["alicorp", "indecopi"]),
}

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))

def leer_feed(xml):
    canal = ET.fromstring(xml).find("channel")
    noticias = []
    for item in canal.findall("item"):
        fuente = item.find("source")
        medio = fuente.text if fuente is not None else None
        titulo = item.findtext("title") or ""
        if medio and titulo.endswith(" - " + medio):
            titulo = titulo[: -len(medio) - 3]
        fecha = item.findtext("pubDate")
        texto = re.sub(r"\s+", " ", re.sub("<[^>]+>", " ", html.unescape(item.findtext("description") or ""))).strip()
        # La descripción suele repetir título y medio: solo cuenta como resumen lo que quede.
        resumen = texto.replace(titulo, "").replace(medio or "", "").replace("\xa0", " ").strip() or None
        noticias.append({"titulo": titulo, "medio": medio,
                         "medio_url": fuente.get("url") if fuente is not None else None,
                         "fecha": parsedate_to_datetime(fecha) if fecha else None,
                         "enlace": item.findtext("link"), "guid": item.findtext("guid"), "resumen": resumen})
    return canal.findtext("copyright"), noticias

resultados, aviso = {}, None
with httpx.Client(timeout=30, follow_redirects=True) as cliente:
    for etiqueta, (consulta, _) in CASOS.items():
        r = cliente.get(FEED, params={"q": consulta, **PERU})
        if r.status_code != 200 or "xml" not in r.headers.get("content-type", ""):
            print("DETENIDO en", etiqueta, r.status_code, r.headers.get("content-type"))
            break
        aviso, resultados[etiqueta] = leer_feed(r.content)
        time.sleep(4)
print(aviso)

In [ ]:
# 2) Cantidad, fechas y coincidencia en el título
for etiqueta, noticias in resultados.items():
    terminos = CASOS[etiqueta][1]
    aciertos = [n for n in noticias if all(t in normalizar(n["titulo"]) for t in terminos)]
    fechas = [n["fecha"] for n in noticias if n["fecha"]]
    rango = f"{min(fechas):%Y-%m-%d} a {max(fechas):%Y-%m-%d}" if fechas else "-"
    print(f"{etiqueta:28} total={len(noticias):3}  en título={len(aciertos):3}  {rango}")
    print("    medios:", Counter(n["medio"] for n in noticias).most_common(5))

In [ ]:
# 3) Campos llenos y duplicados (sobre todas las noticias)
todas = [n for ns in resultados.values() for n in ns]
for campo in todas[0] if todas else []:
    print(f"{campo:10} {sum(bool(n[campo]) for n in todas)}/{len(todas)}")
print("guid distintos:", len({n["guid"] for n in todas}))
print("enlace directo al medio:", sum("news.google.com" not in n["enlace"] for n in todas))
print("resumen:", [n["resumen"] for n in todas if n["resumen"]][:3])

In [ ]:
# 4) Muestra
for etiqueta, noticias in resultados.items():
    print("\n##", etiqueta)
    for n in noticias[:6]:
        print(f"  {n['fecha']:%Y-%m-%d} | {n['medio']} | {n['titulo'][:110]}")